In [0]:
spark.conf.set("spark.sql.shuffle.partitions", "auto")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
# spark.conf.set("spark.scheduler.mode", "FAIR")

In [0]:
# Databricks notebook source
#!/usr/bin/env python3
"""
run_pipeline.py - SQL Recommendation Pipeline Orchestrator (Task-Based)

Refactored for Databricks Workflows where each layer runs as a separate task.
Each main_layer_X() function can be invoked independently via the 'layer' widget.

Usage in Databricks Workflows:
  - Create a task per layer/file
  - Set the 'layer' widget to the appropriate value (e.g., "0", "3_atbs_1")
  - All tasks share the same reference_date, catalog, table_prefix parameters
  - Layer 3 files run as separate tasks for parallel execution on different clusters
"""
# === Databricks Widgets for Task Parameterization ===
dbutils.widgets.text("reference_date", "predict")
dbutils.widgets.dropdown("catalog", "ds_sandbox", ["ds_sandbox", "warehouse"])
dbutils.widgets.text("table_prefix", "next_uk_nextAds_predict_prod")
dbutils.widgets.dropdown("layer", "5", [
    "all", "0", "1", "2", "4", "5",
    "3_atbs_1", "3_atbs_5",
    "3_baskets_1", "3_baskets_5",
    "3_views_1", "3_views_5", "0-3"
])
dbutils.widgets.dropdown("dry_run", "false", ["true", "false"])

In [0]:

import os
import glob
from datetime import datetime, timedelta
import calendar
import argparse
from concurrent.futures import ThreadPoolExecutor
from pyspark.sql import functions as F
from pyspark.sql.window import Window
# ============= CONFIGURATION =============
CONFIG = {
    # Lookback windows (days before reference_date)
    "lookback_windows": {
        "views": 30,      # Views: 30 days
        "atbs": 30,       # ATBs: 30 days (same as views)
        "baskets": 365,   # Baskets: 365 days
    },
}

SQL_PATH = "/Workspace/Users/adrienne_lowe@next.co.uk/next-ads/hackathon_model/sql/"


In [0]:


# ============= HELPER FUNCTIONS =============

def calculate_target_month(reference_date_str):
    """
    Calculate target month date range (same month as reference_date)

    Args:
        reference_date_str: "2025-09-01"

    Returns:
        dict with target month dates
    """
    reference_date = datetime.strptime(reference_date_str, "%Y-%m-%d")

    # First day of
    target_start = reference_date + timedelta(days=1)

    # Last day of 
    target_end = reference_date + timedelta(days=31)

    return {
        "target_month_start": target_start.strftime("%Y-%m-%d"),
        "target_month_end": target_end.strftime("%Y-%m-%d"),
    }


def calculate_custom_date_range(reference_date_str, lookback,days_lag = 0):
    """
    Calculate start and end dates based on reference_date and lookback.
    If reference_date is today, end_date = reference_date; else end_date = reference_date - 1.

    Args:
        reference_date_str: "YYYY-MM-DD"
        lookback: int (number of days)
        days_lag: int (number of days to lag end_date before reference date. Zero by default)

    Returns:
        dict with start_date and end_date
    """
    reference_date = datetime.strptime(reference_date_str, "%Y-%m-%d")
    today = datetime.today().date()
    start_date = (reference_date - timedelta(days=lookback)).strftime("%Y-%m-%d")
    if reference_date.date() == today:
        end_date = reference_date.strftime("%Y-%m-%d")
    else:
        end_date = (reference_date - timedelta(days=days_lag)).strftime("%Y-%m-%d")
    return {
        "start_date": start_date,
        "end_date": end_date,
    }


def validate_dates(reference_date_str, target_dates, calculated_dates):
    """Run date validation checks before pipeline execution"""
    issues = []

    # Check 1: start_date < end_date for all date ranges
    for name, (start, end) in calculated_dates.items():
        if start >= end:
            issues.append(f"FAIL: {name} start_date ({start}) >= end_date ({end})")
        else:
            print(f"PASS: {name} start_date ({start}) < end_date ({end})")

    # # Check 2: target month matches reference_date month
    # reference_date = datetime.strptime(reference_date_str, "%Y-%m-%d")
    # target_start = datetime.strptime(target_dates["target_month_start"], "%Y-%m-%d")
    # if reference_date.month != target_start.month or reference_date.year != target_start.year:
    #     issues.append(f"FAIL: target month ({target_start.month}) != reference_date month ({reference_date.month})")

    # Check 3: BASKETS_BASE_END_DATE covers full target month
    if target_dates["target_month_end"] < reference_date_str:
        issues.append(f"FAIL: target_month_end ({target_dates['target_month_end']}) < reference_date ({reference_date_str})")
    print(f"target start_date ({target_dates['target_month_start']}), target end_date ({target_dates['target_month_end']})")
    return issues


def add_freq12_norm(df, freq_col):
    """Add min-max normalized freq12 column per account_number, reference_date"""
    window = Window.partitionBy("account_number", "reference_date")

    min_val = F.min(freq_col).over(window)
    max_val = F.max(freq_col).over(window)

    return df.withColumn(
        "freq12_norm",
        F.when(max_val == min_val, F.lit(1.0))
        .otherwise((F.col(freq_col) - min_val) / (max_val - min_val))
    )


# Post-process function registry (replaces lambdas for serializability)
POST_PROCESS_REGISTRY = {
    "freq12_norm_atbs": lambda df: add_freq12_norm(df, "theme_clean2_atbs_freq12"),
    "freq12_norm_baskets": lambda df: add_freq12_norm(df, "theme_clean2_baskets_freq12"),
    "freq12_norm_views": lambda df: add_freq12_norm(df, "theme_clean2_views_freq12"),
}


def get_post_process_func(name):
    """Get post-process function by name. Returns None if not found."""
    if name is None:
        return None
    return POST_PROCESS_REGISTRY.get(name)


def execute_sql_file(entry, common_params, catalog, dry_run=False, sql_path=SQL_PATH):
    """Execute SQL and write results to table using Spark DataFrame API.

    Note: OPTIMIZE is NOT run here - call run_optimize_for_entries() at the end of layer execution.
    """
    # Read and render SQL
    with open(sql_path + entry["file"]) as f:
        sql_content = f.read()
    params = entry.get("params") or {}
    rendered_sql = sql_content.format(**params, **common_params)

    # Get table config
    table_name = entry.get("table_name", "").format(**common_params)
    write_mode = entry.get("write_mode")
    partition_by = entry.get("partition_by")
    post_process_name = entry.get("post_process")  # String key for registry lookup

    # If no table_name specified, execute SQL directly (e.g., for views)
    if not table_name:
        if dry_run:
            print(f"-- Executing: {entry['file']}")
            print(rendered_sql)
        else:
            spark.sql(rendered_sql)
            print(f"Executed {entry['file']}")
        return

    full_table_name = f"{catalog}.{table_name}"

    if dry_run:
        print(f"-- Would write to: {full_table_name} (mode={write_mode})")
        print(rendered_sql)
    else:
        # Execute query and get DataFrame
        try:
            df = spark.sql(rendered_sql)
        except Exception:
            print(rendered_sql)
            raise

        # Apply post-processing if specified (lookup from registry)
        post_process_func = get_post_process_func(post_process_name)
        if post_process_func:
            df = post_process_func(df)
            print(f"Applied post-processing '{post_process_name}' to {entry['file']}")

        # Write to table
        writer = df.write.mode(write_mode)
        if write_mode == "overwrite":
            writer = writer.option("overwriteSchema", "true")
        if partition_by:
            writer = writer.partitionBy(*partition_by)
        writer.saveAsTable(full_table_name)
        print(f"Wrote to {full_table_name} ({write_mode})")

In [0]:
def run_optimize_for_entries(entries, common_params, catalog, dry_run=False):
    """Run OPTIMIZE + ZORDER for all entries that have optimize=True.

    Call this at the END of layer execution, after all writes are complete.
    This batches OPTIMIZE operations for better performance.
    """
    print("--- Running OPTIMIZE for layer ---")
    for entry in entries:
        if not entry.get("optimize"):
            continue

        table_name = entry.get("table_name", "").format(**common_params)
        if not table_name:
            continue  # Skip views (no table_name)

        full_table_name = f"{catalog}.{table_name}"
        zorder_cols = entry.get("zorder_by", [])
        optimize_where = entry.get("optimize_where", "").format(**common_params)

        if optimize_where:
            optimize_sql = f"OPTIMIZE {full_table_name} WHERE {optimize_where}"
        else:
            optimize_sql = f"OPTIMIZE {full_table_name}"

        if zorder_cols:
            optimize_sql += f" ZORDER BY ({', '.join(zorder_cols)})"

        if dry_run:
            print(f"-- {optimize_sql}")
        else:
            spark.sql(optimize_sql)
            print(f"Optimized {full_table_name}")


def run_vacuum(catalog, table_prefix, retention_hours=168):
    """Run VACUUM on all tables (typically as a separate scheduled job)"""
    tables = [
        f"{table_prefix}_atbs",
        f"{table_prefix}_baskets",
        f"{table_prefix}_views",
        f"{table_prefix}_atbs_themes",
        f"{table_prefix}_baskets_themes",
        f"{table_prefix}_views_themes",
        f"{table_prefix}_atbs_bytheme",
        f"{table_prefix}_baskets_bytheme",
        f"{table_prefix}_views_bytheme",
    ]
    for table in tables:
        full_name = f"{catalog}.{table}"
        spark.sql(f"VACUUM {full_name} RETAIN {retention_hours} HOURS")
        print(f"Vacuumed {full_name}")

def execute_sql_entry(entry, common_params, catalog, dry_run=False, sql_path=SQL_PATH):
    """
    Wraps execute_sql_file and run_optimize_for_entries into a single 
    unit of work for a specific entry.
    """
    execute_sql_file(entry, common_params, catalog, dry_run, sql_path)
    
    # Run optimize immediately after the specific table is written
    if entry.get("optimize") and not dry_run:
        run_optimize_for_entries([entry], common_params, catalog, dry_run)

def run_layer_parallel(entries, common_params, catalog, dry_run, max_workers=4):
    """
    Executes a list of SQL entries in parallel using a ThreadPool.
    Spark handles the actual resource allocation across the cluster.
    """
    print(f"--- Running {len(entries)} tasks in parallel (max_workers={max_workers}) ---")
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(execute_sql_entry, entry, common_params, catalog, dry_run) 
            for entry in entries
        ]
        # Wait for all tasks in this layer to finish before returning
        for future in futures:
            future.result()

In [0]:


# COMMAND ----------

# ============= BUILD COMMON PARAMS AND SQL FILES DICT =============

def build_common_params(reference_date, catalog, table_prefix):
    """Build the common_params dict from explicit parameters."""
    target_dates = calculate_target_month(reference_date)
    return {
        "catalog": catalog,
        "table_prefix": table_prefix,
        "reference_date": reference_date,
        "target_month_start": target_dates["target_month_start"],
        "target_month_end": target_dates["target_month_end"],
    }


def build_sql_files_dict(reference_date):
    """
    Build the sql_files_dict structure based on reference_date.
    This is called by each main function to get the correct date ranges.
    """
    lookback_windows = CONFIG["lookback_windows"]
    target_dates = calculate_target_month(reference_date)

    start_date_views = calculate_custom_date_range(reference_date, lookback_windows["views"])["start_date"]
    end_date_views = calculate_custom_date_range(reference_date, lookback_windows["views"])["end_date"]
    end_date_views_ly = (datetime.strptime(end_date_views, "%Y-%m-%d") - timedelta(days=365)).strftime("%Y-%m-%d")
    start_date_views_ly = (datetime.strptime(end_date_views_ly, "%Y-%m-%d") - timedelta(days=30)).strftime("%Y-%m-%d")
    start_date_atbs = calculate_custom_date_range(reference_date, lookback_windows["atbs"])["start_date"]
    end_date_atbs = calculate_custom_date_range(reference_date, lookback_windows["atbs"])["end_date"]
    start_date_baskets = calculate_custom_date_range(reference_date, lookback_windows["baskets"])["start_date"]
    end_date_baskets = calculate_custom_date_range(reference_date, lookback_windows["baskets"])["end_date"]
    end_date_baskets_ly = (datetime.strptime(end_date_baskets, "%Y-%m-%d") - timedelta(days=365)).strftime("%Y-%m-%d")
    start_date_baskets_ly = (datetime.strptime(end_date_baskets_ly, "%Y-%m-%d") - timedelta(days=30)).strftime("%Y-%m-%d")

    # BASE_START_DATE = min(start_date_views, start_date_atbs, start_date_baskets)
    # BASE_END_DATE = target_dates["target_month_end"]

    return {
        0: [
            {
                "file": "0_atbs.sql",
                "table_name": "{table_prefix}_atbs",
                "write_mode": "overwrite",
                "optimize": True,
                "zorder_by": ["AccountNumber_RPID", "date"],
                "params": {"start_date": start_date_atbs, "end_date": end_date_atbs},
            },
            # {
            #     "file": "0_base.sql",
            #     # No table_name - creates TEMPORARY VIEW, executed directly
            #     "params": {"start_date": BASE_START_DATE, "end_date": BASE_END_DATE},
            # },
            {
                "file": "0_baskets.sql",
                "table_name": "{table_prefix}_baskets",
                "write_mode": "overwrite",
                "optimize": True,
                "zorder_by": ["account_number", "order_date"],
                "params": {"start_date": start_date_baskets, "end_date": target_dates['target_month_end']},
            },
            {
                "file": "0_baskets_ly.sql",
                "table_name": "{table_prefix}_baskets_ly",
                "write_mode": "overwrite",
                "optimize": True,
                "zorder_by": ["account_number", "order_date"],
                "params": {"start_date": start_date_baskets_ly, "end_date": end_date_baskets_ly},
            },
            {
                "file": "0_theme_mapping.sql",
                # No table_name - creates TEMPORARY VIEW, executed directly
                "params": {},
            },
            {
                "file": "0_views.sql",
                "table_name": "{table_prefix}_views",
                "write_mode": "overwrite",
                "optimize": True,
                "zorder_by": ["AccountNumber_RPID", "date"],
                "params": {"start_date": start_date_views, "end_date": end_date_views},
            },
            {
                "file": "0_views_ly.sql",
                "table_name": "{table_prefix}_views_ly",
                "write_mode": "overwrite",
                "optimize": True,
                "zorder_by": ["AccountNumber_RPID", "date"],
                "params": {"start_date": start_date_views_ly, "end_date": end_date_views_ly},
            },
        ],
        1: [
            {
                "file": "1_atbs_themes.sql",
                "table_name": "{table_prefix}_atbs_themes",
                "write_mode": "overwrite",
                "partition_by": ["reference_date"],
                "optimize": True,
                "zorder_by": ["account_number", "theme_clean"],
                "optimize_where": "reference_date = date'{reference_date}'",
                "params": {"start_date": start_date_atbs, "end_date": end_date_atbs},
            },
            {
                "file": "1_baskets_themes.sql",
                "table_name": "{table_prefix}_baskets_themes",
                "write_mode": "overwrite",
                "partition_by": ["reference_date"],
                "optimize": True,
                "zorder_by": ["account_number", "theme_clean"],
                "optimize_where": "reference_date = date'{reference_date}'",
                "params": {"start_date": start_date_baskets, "end_date": end_date_baskets},
            },
            {
                "file": "1_views_themes.sql",
                "table_name": "{table_prefix}_views_themes",
                "write_mode": "overwrite",
                "partition_by": ["reference_date"],
                "optimize": True,
                "zorder_by": ["account_number", "theme_clean"],
                "optimize_where": "reference_date = date'{reference_date}'",
                "params": {"start_date": start_date_views, "end_date": end_date_views},
            },
            {
                "file": "1a_vatb.sql",
                "table_name": "{table_prefix}_vatb",
                "write_mode": "overwrite",
                "partition_by": ["reference_date"],
                "optimize": True,
                "zorder_by": ["theme_clean1", "theme_clean2"],
                "optimize_where": "reference_date = date'{reference_date}'",
                "params": None,
            },
        ],
        2: [
            {
                "file": "2_advanced_features.sql",
                "table_name": "{table_prefix}_advanced_features",
                "write_mode": "overwrite",
                "partition_by": ["reference_date"],
                "optimize": True,
                "optimize_where": "reference_date = date'{reference_date}'",
                "params": {"start_date": start_date_views, "end_date": end_date_views},
            },
            {
                "file": "2_atbs_bythemes.sql",
                "table_name": "{table_prefix}_atbs_bytheme",
                "write_mode": "overwrite",
                "partition_by": ["reference_date"],
                "optimize": True,
                "zorder_by": ["account_number", "theme_clean"],
                "optimize_where": "reference_date = date'{reference_date}'",
                "params": {"end_date": end_date_views},
            },
            {
                "file": "2_baskets_bythemes.sql",
                "table_name": "{table_prefix}_baskets_bytheme",
                "write_mode": "overwrite",
                "partition_by": ["reference_date"],
                "optimize": True,
                "zorder_by": ["account_number", "theme_clean"],
                "optimize_where": "reference_date = date'{reference_date}'",
                "params": {"end_date": end_date_views},
            },
            {
                "file": "2_views_bythemes.sql",
                "table_name": "{table_prefix}_views_bytheme",
                "write_mode": "overwrite",
                "partition_by": ["reference_date"],
                "optimize": True,
                "zorder_by": ["account_number", "theme_clean"],
                "optimize_where": "reference_date = date'{reference_date}'",
                "params": {"end_date": end_date_views},
            },
            {
                "file": "2_repurchased.sql",
                "table_name": "{table_prefix}_repurchase",
                "write_mode": "overwrite",
                "partition_by": ["reference_date"],
                "optimize": True,
                "zorder_by": ["account_number", "theme_clean"],
                "optimize_where": "reference_date = date'{reference_date}'",
                "params": None,
            },
            {
                "file": "2_target.sql",
                "table_name": "{table_prefix}_baskets_target",
                "write_mode": "overwrite",
                "partition_by": ["reference_date"],
                "optimize": True,
                "zorder_by": ["account_number", "theme_clean"],
                "optimize_where": "reference_date = date'{reference_date}'",
                "params": {"start_date": target_dates['target_month_start'], "end_date": target_dates['target_month_end']},
            },
        ],
        3: [
            {
                "file": "3_atbs_1_algo.sql",
                "table_name": "{table_prefix}_algo_atbs1",
                "write_mode": "overwrite",
                "partition_by": ["reference_date"],
                "optimize": True,
                "zorder_by": ["account_number"],
                "optimize_where": "reference_date = date'{reference_date}'",
                "post_process": "freq12_norm_atbs",  
                "params": None,
            },
            {
                "file": "3_atbs_5_algo.sql",
                "table_name": "{table_prefix}_algo_atbs5",
                "write_mode": "overwrite",
                "partition_by": ["reference_date"],
                "optimize": True,
                "zorder_by": ["account_number"],
                "optimize_where": "reference_date = date'{reference_date}'",
                "post_process": "freq12_norm_atbs",  
                "params": None,
            },
            {
                "file": "3_baskets_1_algo.sql",
                "table_name": "{table_prefix}_algo_baskets1",
                "write_mode": "overwrite",
                "partition_by": ["reference_date"],
                "optimize": True,
                "zorder_by": ["account_number"],
                "optimize_where": "reference_date = date'{reference_date}'",
                "post_process": "freq12_norm_baskets",  
                "params": None,
            },
            {
                "file": "3_baskets_5_algo.sql",
                "table_name": "{table_prefix}_algo_baskets5",
                "write_mode": "overwrite",
                "partition_by": ["reference_date"],
                "optimize": True,
                "zorder_by": ["account_number"],
                "optimize_where": "reference_date = date'{reference_date}'",
                "post_process": "freq12_norm_baskets",  
                "params": None,
            },
            {
                "file": "3_views_1_algo.sql",
                "table_name": "{table_prefix}_algo_views1",
                "write_mode": "overwrite",
                "partition_by": ["reference_date"],
                "optimize": True,
                "zorder_by": ["account_number"],
                "optimize_where": "reference_date = date'{reference_date}'",
                "post_process": "freq12_norm_views",  
                "params": None,
            },
            {
                "file": "3_views_5_algo.sql",
                "table_name": "{table_prefix}_algo_views5",
                "write_mode": "overwrite",
                "partition_by": ["reference_date"],
                "optimize": True,
                "zorder_by": ["account_number"],
                "optimize_where": "reference_date = date'{reference_date}'",
                "post_process": "freq12_norm_views",  
                "params": None,
            },
        ],
        4: [
            {
                "file": "4_customer_features.sql",
                "table_name": "{table_prefix}_customer_features",
                "write_mode": "overwrite",
                "optimize": True,
                "zorder_by": ["account_number"],
                "params": {"start_date": start_date_baskets, "end_date": end_date_baskets},
            },
            {
                "file": "4_customer_segments.sql",
                "table_name": "{table_prefix}_customer_segments",
                "write_mode": "overwrite",
                "optimize": True,
                "zorder_by": ["account_number"],
                "params": {"start_date": start_date_baskets, "end_date": end_date_baskets},
            },
            {
                "file": "4_popularity_metrics.sql",
                "table_name": "{table_prefix}_popularity_metrics",
                "write_mode": "overwrite",
                "optimize": True,
                "zorder_by": ["theme_clean"],
                "params": {"end_date_views_ly": end_date_views_ly,
                           "end_date_baskets_ly": end_date_baskets_ly,
                           "end_date_views": end_date_views},
            },
        ],
        5: [
            {
                "file": "5_spine.sql",
                # No table_name - creates TEMPORARY VIEW, executed directly
                "params": None,
            },
            {
                "file": "6_master_assoc.sql",
                "table_name": "{table_prefix}_master",
                "write_mode": "append",
                "partition_by": ["reference_date"],
                "optimize": True,
                "zorder_by": ["account_number"],
                "optimize_where": "reference_date = date'{reference_date}'",
                "params": {"start_date_view": start_date_views, "end_date_view": end_date_views,
                           "start_date_atbs": start_date_atbs, "end_date_atbs": end_date_atbs,
                           "start_date_bask": start_date_baskets, "end_date_bask": end_date_baskets,
                           "start_date_target": target_dates['target_month_start'], "end_date_target": target_dates['target_month_end']},
            },
        ],
    }


def validate_and_get_dates(reference_date):
    """Validate dates and return calculated date ranges. Raises ValueError if validation fails."""
    lookback_windows = CONFIG["lookback_windows"]
    target_dates = calculate_target_month(reference_date)

    start_date_views = calculate_custom_date_range(reference_date, lookback_windows["views"])["start_date"]
    end_date_views = calculate_custom_date_range(reference_date, lookback_windows["views"])["end_date"]
    start_date_atbs = calculate_custom_date_range(reference_date, lookback_windows["atbs"])["start_date"]
    end_date_atbs = calculate_custom_date_range(reference_date, lookback_windows["atbs"])["end_date"]
    start_date_baskets = calculate_custom_date_range(reference_date, lookback_windows["baskets"])["start_date"]
    end_date_baskets = calculate_custom_date_range(reference_date, lookback_windows["baskets"])["end_date"]

    calculated_dates = {
        "views": (start_date_views, end_date_views),
        "atbs": (start_date_atbs, end_date_atbs),
        "baskets": (start_date_baskets, end_date_baskets),
    }

    issues = validate_dates(reference_date, target_dates, calculated_dates)
    if issues:
        print("\n".join(issues))
        raise ValueError("Date validation failed")

    return calculated_dates

In [0]:


# COMMAND ----------

# ============= MAIN LAYER FUNCTIONS =============

def main_layer_0(reference_date, catalog, table_prefix, sql_files_dict, common_params, dry_run=False):    
    """
    Execute all Layer 0 SQL files (base data extraction).

    Files: 0_atbs.sql, 0_baskets.sql, 0_views.sql
    """
    print(f"=== Starting Layer 0 | reference_date={reference_date} | catalog={catalog} ===")

    entries = sql_files_dict[0]
    # Execute in parallel to save time
    run_layer_parallel(entries, common_params, catalog, dry_run, max_workers=3)
    print(f"=== Completed Layer 0 ===")


def main_layer_1(reference_date, catalog, table_prefix, sql_files_dict, common_params, dry_run=False):
    """
    Execute all Layer 1 SQL files (theme aggregations).

    Files: 1_atbs_themes.sql, 1_baskets_themes.sql, 1_views_themes.sql, 1a_vatb.sql
    """
    print(f"=== Starting Layer 1 | reference_date={reference_date} | catalog={catalog} ===")

    entries = sql_files_dict[1]
    # Execute in parallel to save time
    run_layer_parallel(entries, common_params, catalog, dry_run, max_workers=3)

    print(f"=== Completed Layer 1 ===")


def main_layer_2(reference_date, catalog, table_prefix, sql_files_dict, common_params, dry_run=False):
    """
    Execute all Layer 2 SQL files (advanced features & aggregations).

    Files: 2_advanced_features.sql, 2_atbs_bythemes.sql, 2_baskets_bytheme.sql,
           2_views_bythemes.sql, 2_repurchased.sql, 2_target.sql
    """
    print(f"=== Starting Layer 2 | reference_date={reference_date} | catalog={catalog} ===")

    entries = sql_files_dict[2]
    # Execute in parallel to save time
    run_layer_parallel(entries, common_params, catalog, dry_run, max_workers=3)

    print(f"=== Completed Layer 2 ===")


def main_layer_5(reference_date, catalog, table_prefix, sql_files_dict, common_params, dry_run=False):
    """
    Execute all Layer 5 SQL files (master assembly).

    Files: 5_spine.sql, 6_master_assoc.sql
    """
    print(f"=== Starting Layer 5 | reference_date={reference_date} | catalog={catalog} ===")

    entries = sql_files_dict[5]
    for entry in entries:
        print(f"Processing sequentially: {entry['file']}")
        execute_sql_file(entry, common_params, catalog, dry_run)
        
        # Optimize immediately after each write if requested
        if entry.get("optimize") and not dry_run:
            run_optimize_for_entries([entry], common_params, catalog, dry_run)

    print(f"=== Completed Layer 5 ===")

# COMMAND ----------

# ============= INDIVIDUAL LAYER 3 FUNCTIONS =============
# Each Layer 3 file gets its own function for parallel execution on separate clusters

def main_layer_3_atbs_1(reference_date, catalog, table_prefix, sql_files_dict, common_params, dry_run=False):
    """
    Execute 3_atbs_1_algo.sql only.

    Output: {table_prefix}_algo_atbs1
    """
    print(f"=== Starting Layer 3 atbs_1 | reference_date={reference_date} | catalog={catalog} ===")

    entries = sql_files_dict[3]
    # Execute in parallel to save time
    run_layer_parallel(entries, common_params, catalog, dry_run, max_workers=3)

    print(f"=== Completed Layer 3 atbs_1 ===")


def main_layer_3_atbs_5(reference_date, catalog, table_prefix, sql_files_dict, common_params, dry_run=False):
    """
    Execute 3_atbs_5_algo.sql only.

    Output: {table_prefix}_algo_atbs5
    """
    print(f"=== Starting Layer 3 atbs_5 | reference_date={reference_date} | catalog={catalog} ===")

    entries = sql_files_dict[3]
    # Execute in parallel to save time
    run_layer_parallel(entries, common_params, catalog, dry_run, max_workers=3)

    print(f"=== Completed Layer 3 atbs_5 ===")


def main_layer_3_baskets_1(reference_date, catalog, table_prefix, sql_files_dict, common_params, dry_run=False):
    """
    Execute 3_baskets_1_algo.sql only.

    Output: {table_prefix}_algo_baskets1
    """
    print(f"=== Starting Layer 3 baskets_1 | reference_date={reference_date} | catalog={catalog} ===")

    entries = sql_files_dict[3]
    # Execute in parallel to save time
    run_layer_parallel(entries, common_params, catalog, dry_run, max_workers=3)

    print(f"=== Completed Layer 3 baskets_1 ===")


def main_layer_3_baskets_5(reference_date, catalog, table_prefix, sql_files_dict, common_params, dry_run=False):
    """
    Execute 3_baskets_5_algo.sql only.

    Output: {table_prefix}_algo_baskets5
    """
    print(f"=== Starting Layer 3 baskets_5 | reference_date={reference_date} | catalog={catalog} ===")

    entries = sql_files_dict[3]
    # Execute in parallel to save time
    run_layer_parallel(entries, common_params, catalog, dry_run, max_workers=3)

    print(f"=== Completed Layer 3 baskets_5 ===")


def main_layer_3_views_1(reference_date, catalog, table_prefix, sql_files_dict, common_params, dry_run=False):
    """
    Execute 3_views_1_algo.sql only.

    Output: {table_prefix}_algo_views1
    """
    print(f"=== Starting Layer 3 views_1 | reference_date={reference_date} | catalog={catalog} ===")

    entries = sql_files_dict[3]
    # Execute in parallel to save time
    run_layer_parallel(entries, common_params, catalog, dry_run, max_workers=3)

    print(f"=== Completed Layer 3 views_1 ===")


def main_layer_3_views_5(reference_date, catalog, table_prefix, sql_files_dict, common_params, dry_run=False):
    """
    Execute 3_views_5_algo.sql only.

    Output: {table_prefix}_algo_views5
    """
    print(f"=== Starting Layer 3 views_5 | reference_date={reference_date} | catalog={catalog} ===")

    entries = sql_files_dict[3]
    # Execute in parallel to save time
    run_layer_parallel(entries, common_params, catalog, dry_run, max_workers=3)

    print(f"=== Completed Layer 3 views_5 ===")

# COMMAND ----------

def main_layer_4(reference_date, catalog, table_prefix, sql_files_dict, common_params, dry_run=False):
    """
    Execute all Layer 1 SQL files (theme aggregations).

    Files: 4_customer_features.sql, 4_customer_segments.sql
    """
    print(f"=== Starting Layer 4 | reference_date={reference_date} | catalog={catalog} ===")

    entries = sql_files_dict[4]
    # Execute in parallel to save time
    run_layer_parallel(entries, common_params, catalog, dry_run, max_workers=3)

    print(f"=== Completed Layer 4 ===")

# ============= MAIN ALL FUNCTION =============

def main_all(reference_date, catalog, table_prefix, sql_files_dict, common_params, dry_run=False):
    """
    Execute all layers sequentially.

    Order: Layer 0 -> Layer 1 -> Layer 2 -> Layer 3 (all files) -> Layer 5
    """
    print(f"=== Starting Full Pipeline | reference_date={reference_date} | catalog={catalog} ===")

    # Execute layers in order
    main_layer_0(reference_date, catalog, table_prefix, sql_files_dict, common_params, dry_run)
    main_layer_1(reference_date, catalog, table_prefix, sql_files_dict, common_params, dry_run)
    main_layer_2(reference_date, catalog, table_prefix, sql_files_dict, common_params, dry_run)

    # Layer 3 files (sequentially when running all)
    main_layer_3_atbs_1(reference_date, catalog, table_prefix, sql_files_dict, common_params, dry_run)
    main_layer_3_atbs_5(reference_date, catalog, table_prefix, sql_files_dict, common_params, dry_run)
    main_layer_3_baskets_1(reference_date, catalog, table_prefix, sql_files_dict, common_params, dry_run)
    main_layer_3_baskets_5(reference_date, catalog, table_prefix, sql_files_dict, common_params, dry_run)
    main_layer_3_views_1(reference_date, catalog, table_prefix, sql_files_dict, common_params, dry_run)
    main_layer_3_views_5(reference_date, catalog, table_prefix, sql_files_dict, common_params, dry_run)

    main_layer_4(reference_date, catalog, table_prefix, sql_files_dict, common_params, dry_run)
    main_layer_5(reference_date, catalog, table_prefix, sql_files_dict, common_params, dry_run)

    print(f"=== Completed Full Pipeline ===")

reference_date = dbutils.widgets.get("reference_date")
catalog = dbutils.widgets.get("catalog")
table_prefix = dbutils.widgets.get("table_prefix")
layer = dbutils.widgets.get("layer")
dry_run = dbutils.widgets.get("dry_run") == "true"

from datetime import datetime, timedelta

if reference_date!= 'predict':
    if (datetime.today() - datetime.strptime(reference_date, "%Y-%m-%d")).days < 28:
        raise ValueError("reference_date must be at least 28 days ago to obtain enough data in training set")
else:
    reference_date = datetime.today().strftime("%Y-%m-%d")

print(f"Parameters: reference_date={reference_date}, catalog={catalog}, table_prefix={table_prefix}, layer={layer}, dry_run={dry_run}")

try:
    # Validate and calculate dates once at the start
    calculated_dates = validate_and_get_dates(reference_date)
    
    # Build the parameters and dictionary once
    common_params = build_common_params(reference_date, catalog, table_prefix)
    sql_files_dict = build_sql_files_dict(reference_date)
    
    print(f"Successfully initialized metadata for reference_date: {reference_date}")
except Exception as e:
    print(f"Initialization Failed: {str(e)}")
    raise

# Route to appropriate function based on layer widget
if layer == "all":
    main_all(reference_date, catalog, table_prefix, sql_files_dict, common_params, dry_run)
elif layer == "0-3":
    main_layer_0(reference_date, catalog, table_prefix, sql_files_dict, common_params, dry_run)
    main_layer_1(reference_date, catalog, table_prefix, sql_files_dict, common_params, dry_run)
    main_layer_2(reference_date, catalog, table_prefix, sql_files_dict, common_params, dry_run)
    run_layer_parallel(sql_files_dict[3], common_params, catalog, dry_run, max_workers=3)
elif layer == "4":
    main_layer_4(reference_date, catalog, table_prefix, sql_files_dict, common_params, dry_run)
elif layer == "5":
    main_layer_5(reference_date, catalog, table_prefix, sql_files_dict, common_params, dry_run)
else:
    raise ValueError(f"Unknown layer: {layer}. Valid options: all, 0, 1, 2, 4, 5, 3_atbs_1, 3_atbs_5, 3_baskets_1, 3_baskets_5, 3_views_1, 3_views_5")


In [0]:
if layer == "5":
    predict_df = spark.read.table("ds_sandbox.next_uk_nextAds_predict_prod_master").filter(F.col('rundate') == F.current_date()).distinct()
    month_value = F.month(F.date_add(F.col("reference_date"), 1))
    predict_df = predict_df.withColumn("month", F.lit(month_value))
else:
    print('skipped as awating final layer')

In [0]:
if layer == "5":
    # perform seasons filtering and any other
    full_df = predict_df#.withColumn('seasons_cust', F.when(F.col('seasons_cust') == 'seasons', 1).otherwise(0))
    # print('filtered for seasons')

    decimal_cols = [
        'repurchase_ratio', 
        # 'user_velocity_score', 
        'Familyconfidence_score', 'Coupleconfidence_score',
        'Womenswearconfidence_score', 'Menswearconfidence_score', 'Beautyconfidence_score', 'Homeconfidence_score'
    ]

    # Cast them to double before calling toPandas
    base = full_df.select([
        F.col(c).cast("double") if c in decimal_cols else F.col(c) 
        for c in full_df.columns
    ])
    display(base.groupby(F.col('label')).count())
    print('optimised base')
    base.cache().count()

    # save base table
    base.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("ds_sandbox.next_uk_nextAds_predict_prod_complete")
else:
    print('skipped as awating final layer')